# URL Classifier — Fixed Version
### Why `google.com` was classified as phishing — and how this fixes it

**Root causes fixed:**
1. URL normalization — strips `http://`, `https://`, `www.` so bare domains aren't treated differently
2. Domain/path split — TF-IDF trained separately on domain vs path (they mean different things)
3. Tranco top-1M whitelist feature — famous domains get a strong 'benign' signal
4. Class weight balancing — stops the model defaulting to 'phishing' when uncertain
5. Decision threshold tuning — moves the phishing threshold up to reduce false positives
6. Better structural features — TLD suspiciousness, brand impersonation, IP-in-URL, entropy

## 0. Setup

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q xgboost lightgbm scikit-learn tqdm requests

In [ ]:
import pandas as pd
import numpy as np
import math
import re
from collections import Counter
from urllib.parse import urlparse
from tqdm.auto import tqdm
from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.calibration import CalibratedClassifierCV

import xgboost as xgb
import seaborn as sns
import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

## 1. Load Data

In [ ]:
# Upload your CSV or mount Drive
# from google.colab import files; files.upload()
# from google.colab import drive; drive.mount('/content/drive')

CSV_PATH = 'urls.csv'

df = pd.read_csv(CSV_PATH)[['url', 'type']].dropna()
df['url'] = df['url'].astype(str).str.strip()

print(f"Total: {len(df):,} samples")
print(df['type'].value_counts())
print("\nFirst few rows:")
print(df.head())

## 2. URL Normalization — THE KEY FIX

This is the most important fix. Without normalization:
- `google.com` → model has never seen this exact format as benign
- `https://google.com` → model treats this as completely different
- `http://free-win.xyz` → model has seen this format in phishing

After normalization, all three are compared on the same playing field.

In [ ]:
def normalize_url(url: str) -> str:
    """Normalize URL to a consistent format for the model."""
    url = url.strip().lower()
    # Remove protocol
    url = re.sub(r'^https?://', '', url)
    # Remove www.
    url = re.sub(r'^www\.', '', url)
    # Remove trailing slash (only root)
    if url.endswith('/') and url.count('/') == 1:
        url = url.rstrip('/')
    return url


def split_url(url: str):
    """Split URL into (domain, path+query) parts after normalization."""
    url = normalize_url(url)
    parts = url.split('/', 1)
    domain = parts[0]
    path   = parts[1] if len(parts) > 1 else ''
    return domain, path


# Test the normalization
test_urls = [
    'google.com',
    'https://google.com',
    'http://www.google.com',
    'http://free-gift-card-win-now.xyz',
    'https://paypal.com.secure-login.tk/verify/account',
]

for u in test_urls:
    domain, path = split_url(u)
    print(f"{u}")
    print(f"  domain={domain!r}  path={path!r}")
    print()

In [ ]:
# Apply normalization to dataset
df['url_norm']   = df['url'].apply(normalize_url)
df['url_domain'] = df['url'].apply(lambda u: split_url(u)[0])
df['url_path']   = df['url'].apply(lambda u: split_url(u)[1])

print("Normalized sample:")
print(df[['url', 'url_norm', 'url_domain', 'url_path', 'type']].head(10).to_string())

## 3. Tranco Top-1M Whitelist Feature

The Tranco list is a research-grade ranking of the top 1 million domains. 
If a domain is in the top 10k, it's almost certainly benign (Google, YouTube, GitHub, etc.).
This single feature will fix most false positives like `google.com`.

In [ ]:
import requests
import io
import zipfile

def load_tranco_top_n(n=100000):
    """
    Download Tranco top-1M list and return a dict {domain: rank}.
    We use the top 100k by default — covers all major sites.
    """
    print(f"Downloading Tranco top-{n:,} list...")
    try:
        r = requests.get('https://tranco-list.eu/top-1m.csv.zip', timeout=30)
        z = zipfile.ZipFile(io.BytesIO(r.content))
        with z.open('top-1m.csv') as f:
            lines = f.read().decode().splitlines()

        tranco = {}
        for line in lines[:n]:
            rank, domain = line.strip().split(',', 1)
            tranco[domain.lower()] = int(rank)
        print(f"Loaded {len(tranco):,} domains from Tranco")
        return tranco
    except Exception as e:
        print(f"Could not download Tranco ({e}). Using manual fallback list.")
        # Fallback: hardcoded list of very common benign domains
        fallback = [
            'google.com', 'youtube.com', 'facebook.com', 'twitter.com', 'instagram.com',
            'linkedin.com', 'github.com', 'wikipedia.org', 'amazon.com', 'microsoft.com',
            'apple.com', 'netflix.com', 'reddit.com', 'stackoverflow.com', 'whatsapp.com',
            'tiktok.com', 'bing.com', 'yahoo.com', 'baidu.com', 'zoom.us',
            'dropbox.com', 'spotify.com', 'adobe.com', 'salesforce.com', 'shopify.com',
            'cloudflare.com', 'wordpress.com', 'openai.com', 'anthropic.com', 'stripe.com',
        ]
        return {d: i+1 for i, d in enumerate(fallback)}


TRANCO = load_tranco_top_n(100000)

# Test it
for d in ['google.com', 'github.com', 'free-gift-card.xyz', 'paypal.com.secure-login.tk']:
    rank = TRANCO.get(d, None)
    print(f"  {d}: rank={rank}")

## 4. Feature Engineering

In [ ]:
SUSPICIOUS_TLDS = {
    '.xyz', '.top', '.club', '.online', '.site', '.tk', '.ml',
    '.ga', '.cf', '.gq', '.pw', '.cc', '.info', '.biz', '.ws',
    '.cn', '.ru', '.su', '.to', '.click', '.link', '.download'
}

TRUSTED_TLDS = {'.com', '.org', '.net', '.edu', '.gov', '.io', '.co.uk', '.ac.uk'}

BRAND_KEYWORDS = [
    'paypal', 'apple', 'google', 'amazon', 'microsoft', 'facebook',
    'netflix', 'instagram', 'twitter', 'linkedin', 'bank', 'secure',
    'login', 'signin', 'account', 'verify', 'update', 'confirm',
    'password', 'ebay', 'chase', 'citibank', 'wellsfargo', 'hsbc'
]

def url_entropy(s):
    if not s: return 0
    counts = Counter(s)
    total = len(s)
    return -sum((c/total) * math.log2(c/total) for c in counts.values())

def get_tld(domain):
    parts = domain.rsplit('.', 1)
    return '.' + parts[-1] if len(parts) > 1 else ''

def is_ip_address(domain):
    return bool(re.match(r'^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$', domain))


class URLFeatures(BaseEstimator, TransformerMixin):
    """
    Extract structured features from URLs.
    Input is the ORIGINAL url (normalization happens inside).
    """
    def fit(self, X, y=None): return self

    def transform(self, X):
        features = []
        for url in tqdm(X, desc="URL features", leave=False):
            norm   = normalize_url(url)
            domain, path = split_url(url)
            tld    = get_tld(domain)
            sub_parts = domain.split('.')
            n = normalize_url  # alias

            # --- Tranco rank (THE KEY FEATURE) ---
            tranco_rank = TRANCO.get(domain, None)
            in_top_1k   = int(tranco_rank is not None and tranco_rank <= 1000)
            in_top_10k  = int(tranco_rank is not None and tranco_rank <= 10000)
            in_top_100k = int(tranco_rank is not None and tranco_rank <= 100000)
            tranco_score = 0 if tranco_rank is None else max(0, 1 - (tranco_rank / 100000))

            # --- Length features ---
            full_len    = len(norm)
            domain_len  = len(domain)
            path_len    = len(path)

            # --- Domain structure ---
            subdomain_count = max(0, len(sub_parts) - 2)
            has_many_subs   = int(subdomain_count >= 3)
            # Phishing trick: paypal.com.evil.xyz — benign domain as subdomain
            has_brand_as_sub = int(any(brand in '.'.join(sub_parts[:-2]) for brand in BRAND_KEYWORDS))

            # --- TLD ---
            is_suspicious_tld = int(tld in SUSPICIOUS_TLDS)
            is_trusted_tld    = int(tld in TRUSTED_TLDS)

            # --- Character counts (on normalized URL) ---
            dot_count   = norm.count('.')
            dash_count  = norm.count('-')
            slash_count = norm.count('/')
            at_count    = norm.count('@')   # redirect trick
            pct_count   = norm.count('%')   # URL encoding
            eq_count    = norm.count('=')
            amp_count   = norm.count('&')
            digit_count = sum(c.isdigit() for c in norm)

            # --- Ratios ---
            digit_ratio  = digit_count / max(full_len, 1)
            alpha_ratio  = sum(c.isalpha() for c in norm) / max(full_len, 1)
            special_ratio = (at_count + pct_count + eq_count) / max(full_len, 1)

            # --- Suspicious patterns ---
            has_ip      = int(is_ip_address(domain))
            has_at      = int('@' in norm)
            double_slash = int('//' in norm[2:])  # after domain
            has_hex_enc = int('%' in path)

            # --- Brand keyword count (in full URL) ---
            brand_count = sum(1 for kw in BRAND_KEYWORDS if kw in norm)

            # --- Entropy ---
            full_entropy   = url_entropy(norm)
            domain_entropy = url_entropy(domain)

            # --- Specific length thresholds ---
            long_url    = int(full_len > 75)
            very_long   = int(full_len > 150)

            features.append([
                # Whitelist (most important)
                in_top_1k, in_top_10k, in_top_100k, tranco_score,
                # Domain structure
                domain_len, subdomain_count, has_many_subs, has_brand_as_sub,
                # TLD
                is_suspicious_tld, is_trusted_tld,
                # Lengths
                full_len, path_len, long_url, very_long,
                # Char counts
                dot_count, dash_count, slash_count, at_count,
                pct_count, eq_count, amp_count, digit_count,
                # Ratios
                digit_ratio, alpha_ratio, special_ratio,
                # Suspicious
                has_ip, has_at, double_slash, has_hex_enc,
                # Brand & entropy
                brand_count, full_entropy, domain_entropy,
            ])

        return np.array(features, dtype=np.float32)


print("Feature extractor defined ✅")

In [ ]:
# Quick sanity check on features
demo = URLFeatures()
demo_urls = [
    'google.com',
    'https://google.com',
    'http://paypal.com.secure-login.tk/verify',
    'http://free-gift-card-win-now.xyz',
]
feats = demo.transform(demo_urls)
feat_names = [
    'in_top_1k', 'in_top_10k', 'in_top_100k', 'tranco_score',
    'domain_len', 'subdomain_count', 'has_many_subs', 'has_brand_as_sub',
    'is_suspicious_tld', 'is_trusted_tld',
    'full_len', 'path_len', 'long_url', 'very_long',
    'dot_count', 'dash_count', 'slash_count', 'at_count',
    'pct_count', 'eq_count', 'amp_count', 'digit_count',
    'digit_ratio', 'alpha_ratio', 'special_ratio',
    'has_ip', 'has_at', 'double_slash', 'has_hex_enc',
    'brand_count', 'full_entropy', 'domain_entropy',
]

feat_df = pd.DataFrame(feats, columns=feat_names, index=demo_urls)
# Show the most informative features
key_cols = ['in_top_1k', 'in_top_10k', 'tranco_score', 'is_suspicious_tld', 'has_brand_as_sub', 'full_entropy', 'full_len']
print(feat_df[key_cols].to_string())

## 5. Combined Feature Extractor
Splits TF-IDF into domain-level and path-level — much better signal than treating the full URL as one string.

In [ ]:
class CombinedFeatures(BaseEstimator, TransformerMixin):
    def __init__(self):
        # Char n-grams on the DOMAIN part
        self.tfidf_domain = TfidfVectorizer(
            analyzer='char_wb', ngram_range=(3, 5),
            max_features=8000, sublinear_tf=True
        )
        # Char n-grams on the PATH part
        self.tfidf_path = TfidfVectorizer(
            analyzer='char_wb', ngram_range=(3, 5),
            max_features=4000, sublinear_tf=True
        )
        self.url_features = URLFeatures()

    def fit(self, X, y=None):
        domains = [split_url(u)[0] for u in X]
        paths   = [split_url(u)[1] for u in X]
        self.tfidf_domain.fit(domains)
        self.tfidf_path.fit(paths)
        return self

    def transform(self, X):
        domains = [split_url(u)[0] for u in X]
        paths   = [split_url(u)[1] for u in X]

        f_domain  = self.tfidf_domain.transform(domains)
        f_path    = self.tfidf_path.transform(paths)
        f_manual  = csr_matrix(self.url_features.transform(X))

        return hstack([f_domain, f_path, f_manual])


print("CombinedFeatures defined ✅")

## 6. Label Encoding + Stratified Split

In [ ]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['type'])
print("Labels:", dict(zip(le.classes_, le.transform(le.classes_))))

# Optional: cap per-class for speed (set to None to use all)
SUBSET_PER_CLASS = 10000

if SUBSET_PER_CLASS:
    df_sub = (
        df.groupby('type', group_keys=False)
          .apply(lambda g: g.sample(min(len(g), SUBSET_PER_CLASS), random_state=42))
          .reset_index(drop=True)
    )
else:
    df_sub = df

print(f"\nUsing {len(df_sub):,} samples")
print(df_sub['type'].value_counts())

X = df_sub['url'].values
y = df_sub['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train
)

print(f"\nTrain: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

## 7. Extract Features

In [ ]:
print("Extracting features...")
features = CombinedFeatures()
X_train_f = features.fit_transform(X_train)
X_val_f   = features.transform(X_val)
X_test_f  = features.transform(X_test)
print(f"Feature shape: {X_train_f.shape}")

## 8. Train XGBoost with Class Weights

In [ ]:
# Compute class weights — penalize the majority class less,
# rare classes more. Especially important for phishing.
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight('balanced', y_train)

xgb_params = dict(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=5,
    gamma=0.1,
    eval_metric='mlogloss',
    early_stopping_rounds=25,
    verbosity=0,
)

if DEVICE == 'cuda':
    xgb_params['device'] = 'cuda'
    xgb_params['tree_method'] = 'hist'
    print("XGBoost: GPU mode")
else:
    xgb_params['tree_method'] = 'hist'

model = xgb.XGBClassifier(**xgb_params)
model.fit(
    X_train_f, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_val_f, y_val)],
    verbose=50
)

print(f"Best iteration: {model.best_iteration}")

## 9. Evaluate + Threshold Tuning

Threshold tuning lets you control the tradeoff between:
- **False positives** (benign URLs called phishing — bad for users)
- **False negatives** (phishing URLs called benign — security risk)

The default threshold of 0.5 often isn't optimal. We tune it on the validation set.

In [ ]:
y_pred = model.predict(X_test_f)
print("=== Evaluation (default threshold) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_,
            yticklabels=le.classes_, cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout(); plt.show()

In [ ]:
# === Threshold tuning for phishing ===
# Increase phishing threshold → fewer false positives (google.com won't be called phishing)
# Lower threshold → fewer false negatives (catches more actual phishing)

PHISHING_CLASS_IDX = list(le.classes_).index('phishing')

y_proba_val = model.predict_proba(X_val_f)

# Try different thresholds on validation set
thresholds = np.arange(0.3, 0.9, 0.05)
results = []

for t in thresholds:
    y_pred_t = y_proba_val.argmax(axis=1).copy()
    # Override: only classify as phishing if P(phishing) > threshold
    phishing_proba = y_proba_val[:, PHISHING_CLASS_IDX]
    # For samples predicted as phishing at default, re-check threshold
    for i in range(len(y_pred_t)):
        if y_pred_t[i] == PHISHING_CLASS_IDX and phishing_proba[i] < t:
            # Re-classify as the next most likely class
            alt = np.argsort(y_proba_val[i])[::-1]
            y_pred_t[i] = alt[1]  # second most likely

    acc = accuracy_score(y_val, y_pred_t)
    report = classification_report(y_val, y_pred_t, target_names=le.classes_, output_dict=True)
    phishing_f1 = report.get('phishing', {}).get('f1-score', 0)
    phishing_rec = report.get('phishing', {}).get('recall', 0)
    phishing_pre = report.get('phishing', {}).get('precision', 0)
    results.append((t, acc, phishing_f1, phishing_pre, phishing_rec))

res_df = pd.DataFrame(results, columns=['threshold', 'accuracy', 'phishing_f1', 'phishing_precision', 'phishing_recall'])
print(res_df.to_string(float_format='{:.3f}'.format))

# Pick threshold that maximizes phishing precision (fewer false positives)
best_threshold_row = res_df.sort_values('phishing_precision', ascending=False).iloc[0]
BEST_THRESHOLD = best_threshold_row['threshold']
print(f"\nRecommended threshold: {BEST_THRESHOLD:.2f} (phishing precision={best_threshold_row['phishing_precision']:.3f})")

## 10. Prediction Function with Tuned Threshold

In [ ]:
def predict_url(url: str, threshold: float = BEST_THRESHOLD) -> dict:
    """
    Predict URL type with confidence scores.
    threshold: minimum confidence required to call something 'phishing'
    """
    feat = features.transform([url])
    proba = model.predict_proba(feat)[0]

    # Get top predicted class
    ranked = np.argsort(proba)[::-1]
    pred_idx = ranked[0]

    # Apply threshold for phishing specifically
    if pred_idx == PHISHING_CLASS_IDX and proba[pred_idx] < threshold:
        pred_idx = ranked[1]  # use second-most-likely

    label = le.inverse_transform([pred_idx])[0]

    return {
        'url': url,
        'prediction': label,
        'confidence': f"{proba[pred_idx]:.1%}",
        'scores': {cls: f"{p:.1%}" for cls, p in zip(le.classes_, proba)}
    }


# === TEST CASES ===
test_cases = [
    # Should be benign
    'google.com',
    'https://google.com',
    'https://github.com/user/repo',
    'https://sbi.co.in/portal/web/guest/home',
    'https://www.youtube.com/watch?v=abc123',
    # Should be phishing
    'http://paypal.com.secure-login.tk/verify',
    'http://free-gift-card-win-now.xyz',
    'http://apple-id-verify.ml/account/login',
    'http://192.168.1.1/phishing-page',
]

print(f"{'URL':<55} {'PREDICTION':<12} {'CONFIDENCE':<12} {'P(phishing)':<12}")
print('-' * 95)
for url in test_cases:
    result = predict_url(url)
    phish_score = result['scores'].get('phishing', 'N/A')
    marker = '⚠️' if result['prediction'] in ('phishing', 'malware') else '✅'
    print(f"{url:<55} {result['prediction']:<12} {result['confidence']:<12} {phish_score:<12} {marker}")

## 11. Feature Importance — Understanding what the model learned

In [ ]:
# Get manual feature importances (the interpretable part)
# TF-IDF features come first, manual features are at the end
n_tfidf_domain = features.tfidf_domain.max_features or 8000
n_tfidf_path   = features.tfidf_path.max_features or 4000
n_manual = 32

manual_feat_names = [
    'in_top_1k', 'in_top_10k', 'in_top_100k', 'tranco_score',
    'domain_len', 'subdomain_count', 'has_many_subs', 'has_brand_as_sub',
    'is_suspicious_tld', 'is_trusted_tld',
    'full_len', 'path_len', 'long_url', 'very_long',
    'dot_count', 'dash_count', 'slash_count', 'at_count',
    'pct_count', 'eq_count', 'amp_count', 'digit_count',
    'digit_ratio', 'alpha_ratio', 'special_ratio',
    'has_ip', 'has_at', 'double_slash', 'has_hex_enc',
    'brand_count', 'full_entropy', 'domain_entropy',
]

importances = model.feature_importances_
manual_importances = importances[n_tfidf_domain + n_tfidf_path:
                                  n_tfidf_domain + n_tfidf_path + n_manual]

imp_df = pd.DataFrame({'feature': manual_feat_names, 'importance': manual_importances})
imp_df = imp_df.sort_values('importance', ascending=True).tail(15)

plt.figure(figsize=(8, 5))
plt.barh(imp_df['feature'], imp_df['importance'], color='steelblue')
plt.title('Top 15 Manual Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 12. Save

In [ ]:
import joblib

joblib.dump({
    'model': model,
    'features': features,
    'label_encoder': le,
    'phishing_threshold': BEST_THRESHOLD,
    'phishing_class_idx': PHISHING_CLASS_IDX,
}, 'url_classifier_fixed.pkl')

print("Saved url_classifier_fixed.pkl ✅")
print(f"Phishing threshold: {BEST_THRESHOLD:.2f}")
print("\nTo use later:")
print("  data = joblib.load('url_classifier_fixed.pkl')")
print("  model, features, le = data['model'], data['features'], data['label_encoder']")

# Download from Colab
# from google.colab import files
# files.download('url_classifier_fixed.pkl')